In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

T0 = time.time()
SEED = 20260806
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

IMG = 224
CROP_MM = 160.0
GROUP = 3
N_GROUP = 3
CACHE_SLICES = GROUP * N_GROUP
HDR_THREADS = 16
PIX_THREADS = 12
EVAL_BATCH = 12
TIME_BUDGET = 8.0 * 3600
BACKBONE_VARIANT = "small"
UNFREEZE_LAST = 6
MODEL_FILE = "rsna_20260807_v4.pt"

LAT_FALLBACK        = "auto"
LAT_MIN_AGREEMENT   = 0.85
LAT_MIN_OFFSET_MM   = 5.0

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file() and (cand / "test_series").is_dir():
                    return cand
    raise FileNotFoundError("competition mount not found")


def find_dinov2(variant="small"):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


def find_model_path():
    direct = [
        Path("/kaggle/input/datasets/tonylica/rsna2026-models") / MODEL_FILE,
        Path(MODEL_FILE),
    ]
    for p in direct:
        if p.is_file():
            return p
    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.rglob(MODEL_FILE):
            if p.is_file():
                return p
    raise FileNotFoundError(f"{MODEL_FILE} is not attached")


ROOT = find_root()
log(f"input root: {ROOT}")


In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "ImageLaterality",
            "ImagePositionPatient", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept"]


def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame()
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [ ]:
# --- laterality resolution -----------------------------------------------------------
# The tag is authoritative where it exists. Where it does not, the patient x-coordinate
# can stand in, but only if it agrees with the tag on the studies that have both: a wrong
# flip is worse than no flip, and knee coils often place the joint near isocentre, where
# the sign carries no information at all.

def _tag_side(g):
    v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
    if "ImageLaterality" in g.columns:
        v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
    v = [x[0] for x in v if x and x[0] in ("L", "R")]
    return v[0] if v else None


def _position_side(g):
    xs = []
    for s in g.get("ImagePositionPatient", pd.Series(dtype=object)).dropna():
        try:
            xs.append(float(str(s).split("|")[0]))
        except Exception:
            pass
    if not xs:
        return None
    x = float(np.median(xs))
    if abs(x) < LAT_MIN_OFFSET_MM:
        return None                      # centred in the coil: the sign means nothing
    return "R" if x < 0 else "L"         # LPS: the right knee sits at negative x


def laterality_maps(h):
    """Return (side_by_study, diagnostics). Uses the fallback only if it earns it."""
    tag, pos = {}, {}
    for st, g in h.groupby("StudyInstanceUID"):
        tag[st] = _tag_side(g)
        pos[st] = _position_side(g)

    both = [st for st in tag if tag[st] and pos[st]]
    agree = float(np.mean([tag[st] == pos[st] for st in both])) if both else np.nan
    have_tag = float(np.mean([v is not None for v in tag.values()]))

    if LAT_FALLBACK == "on":
        use = True
    elif LAT_FALLBACK == "off":
        use = False
    else:
        use = bool(both) and np.isfinite(agree) and agree >= LAT_MIN_AGREEMENT

    side = {st: (tag[st] or (pos[st] if use else None)) for st in tag}
    covered = float(np.mean([v is not None for v in side.values()]))
    info = {"tag_coverage": have_tag, "agreement": agree, "n_compared": len(both),
            "fallback_used": use, "final_coverage": covered}
    log(f"laterality: tag on {have_tag:.1%} of studies, x-sign agrees with it on "
        f"{agree:.1%} of {len(both)} comparable studies, fallback "
        f"{'enabled' if use else 'disabled'} -> {covered:.1%} normalised")
    return side, info

In [ ]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand) == 0 and fluid is False:
                # T1 slots are the scarcest; fall back to any non-fat-sat, non-fluid
                # series in the plane before giving up on the slot entirely.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


In [ ]:
def _natural_key(name):
    """Stable fallback for numeric filenames such as 1.dcm, 2.dcm, 10.dcm."""
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def spatially_sorted_files(rec):
    """Sort a DICOM stack by patient-space position, then InstanceNumber.

    Filename order is not an anatomical guarantee and can scramble the three-slice
    pseudo-RGB inputs. The varying ImagePositionPatient axis identifies the stack
    direction without assuming a scanner-specific orientation. Header failures fall
    back to InstanceNumber and finally natural filename order.
    """
    files, directory = list(rec["files"]), rec["dir"]
    rows = []
    for pos, name in enumerate(files):
        ipp, instance = None, None
        try:
            ds = pydicom.dcmread(
                os.path.join(directory, name), stop_before_pixels=True, force=True,
                specific_tags=["ImagePositionPatient", "InstanceNumber"],
            )
            raw_ipp = getattr(ds, "ImagePositionPatient", None)
            if raw_ipp is not None and len(raw_ipp) >= 3:
                candidate = np.asarray(raw_ipp[:3], dtype=np.float64)
                if np.isfinite(candidate).all():
                    ipp = candidate
            raw_instance = getattr(ds, "InstanceNumber", None)
            if raw_instance is not None:
                instance = float(raw_instance)
        except Exception:
            pass
        rows.append((name, ipp, instance, pos))

    positioned = [r for r in rows if r[1] is not None]
    if len(positioned) >= max(2, int(0.8 * len(rows))):
        xyz = np.stack([r[1] for r in positioned])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        fallback = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (
            float(r[1][axis]) if r[1] is not None else fallback,
            r[2] if r[2] is not None else float("inf"),
            r[3],
        ))
    elif sum(r[2] is not None for r in rows) >= max(2, int(0.8 * len(rows))):
        rows.sort(key=lambda r: (
            r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows]


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns float32 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    # was `N_SLICE`, which is never defined: a latent NameError on the default path.
    n_slice = CACHE_SLICES if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = spatially_sorted_files(rec), rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over the central 60% of the stack: the outer slices of a knee
    # series are mostly soft tissue outside the joint.
    lo, hi = int(0.20 * (n - 1)), int(0.80 * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = np.zeros((out_size, out_size), dtype=np.float32)
        planes.append(a)

    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


In [ ]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


In [ ]:
def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    gc.collect()
    return studies, cache, mask


In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. Richer alternatives were measured
    against it and lost: with a study-level label there is no signal telling the model
    which part of a study matters, so extra attention parameters have nothing to learn
    from and spend their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

        # A weak prior keeps noisy study-level supervision from learning obviously
        # implausible view assignments. A value of 0.55 is only a 1.7x softmax tilt,
        # so the learned query can override it when the data disagrees.
        prior = torch.zeros(n_out, n_slot)
        if SLOT_SCHEME == "recovered" and n_slot == 6 and n_out == len(TARGETS):
            preferred = {
                "ACL": (0, 3, 5), "MCL": (1, 4),
                "Medial Meniscus": (0, 1, 3, 4),
                "Lateral Meniscus": (0, 1, 3, 4),
                "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
                "PF OA": (0, 2, 5), "Effusion": (0, 2),
                "Synovitis": (0, 2), "Baker's": (0,),
                "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
            }
            for target, slots in preferred.items():
                prior[TARGETS.index(target), list(slots)] = 0.55
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = (torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
               + self.slot_prior.unsqueeze(0))
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim):
        super().__init__()
        self.backbone = backbone
        self.head = SlotHead(dim, N_SLOT, len(TARGETS))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        # The mean is stable for diffuse disease; top-k pooling preserves focal high
        # responses from tears, contusions and fractures that a global mean dilutes.
        k = max(1, patch.shape[1] // 8)
        focal = patch.topk(k, dim=1).values.mean(1)
        feat = torch.cat([out[:, 0], patch.mean(1), focal], dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


In [ ]:
def build_model():
    """Load the encoder and open the last few blocks for training.

    Only the last UNFREEZE_LAST blocks and the final norm are trainable. The early
    blocks of a self-supervised transformer are generic edge and texture filters; there
    is not enough supervision here to improve them and quite enough to damage them.
    """
    from transformers import AutoModel
    p = find_dinov2(BACKBONE_VARIANT)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - UNFREEZE_LAST):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size * 3
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {UNFREEZE_LAST} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim}")
    return Model(bb, dim)


In [ ]:
def take_group(cache_rows, g):
    """Slice GROUP consecutive channels out of the cached slices."""
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]


# `augment` is defined further down, after the reasoning for dropping the flip.


@torch.no_grad()
def predict(model, cache, mask, idx, dev):
    """Average the logits over the groups of each slot.

    Training sees one group at a time, which acts as augmentation along the stack;
    inference averages over all of them, which is the same aggregation the frozen
    pipeline used and the one that measured best.
    """
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        rows = torch.from_numpy(cache[sel]).to(dev)
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(take_group(rows, g), m).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j])
                             if len(set(y[:, j])) > 1 else np.nan
                             for j in range(y.shape[1])]))


In [ ]:
def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for this inference notebook.")
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    arch = f"sm_{cap[0]}{cap[1]}"
    supported = set(torch.cuda.get_arch_list())
    log(f"cuda device: {name}, capability {arch}; torch supports {sorted(supported)}")
    if arch not in supported:
        raise RuntimeError(f"Kaggle assigned {name} ({arch}), but this PyTorch build cannot run on it.")
    return torch.device("cuda")


def load_bundle():
    path = find_model_path()
    log(f"model bundle: {path}")
    try:
        bundle = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        bundle = torch.load(path, map_location="cpu")
    return bundle


def apply_bundle_config(bundle):
    global TARGETS, SLOTS, N_SLOT, IMG, GROUP, N_GROUP, CACHE_SLICES, BACKBONE_VARIANT
    TARGETS = list(bundle.get("targets", TARGETS))
    SLOTS = [tuple(x) for x in bundle.get("slots", SLOTS)]
    N_SLOT = len(SLOTS)
    IMG = int(bundle.get("img", IMG))
    GROUP = int(bundle.get("group", GROUP))
    N_GROUP = int(bundle.get("n_group", N_GROUP))
    CACHE_SLICES = GROUP * N_GROUP
    variant = str(bundle.get("model_variant", "dinov2-small")).split("-")[-1]
    BACKBONE_VARIANT = "base" if variant == "base" else "small"
    log(f"bundle config: backbone={BACKBONE_VARIANT}, img={IMG}, groups={N_GROUP}, slots={N_SLOT}")


def write_submission(rank_sum, n_models, st_te, test_df):
    P = rank_sum / max(n_models, 1)
    sub = pd.DataFrame(P, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", st_te)
    sub["StudyInstanceUID"] = sub["StudyInstanceUID"].astype(str)
    out = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    out[TARGETS] = out[TARGETS].fillna(0.5)
    out.to_csv("submission.csv", index=False)
    return out


def main():
    dev = require_cuda()
    bundle = load_bundle()
    apply_bundle_config(bundle)

    test_df = pd.read_csv(ROOT / "test.csv")
    test_df["StudyInstanceUID"] = test_df["StudyInstanceUID"].astype(str)
    test_series = pd.read_csv(ROOT / "test_series.csv")
    test_series["StudyInstanceUID"] = test_series["StudyInstanceUID"].astype(str)
    test_series["SeriesInstanceUID"] = test_series["SeriesInstanceUID"].astype(str)
    log(f"test {test_df.shape}; test_series {test_series.shape}")

    plane_map = dict(zip(test_series["SeriesInstanceUID"], test_series["Anatomical_Plane"]))
    log("header pass: test")
    hte = annotate(walk("test_series"))
    log(f"  {len(hte)} test series")
    lat_te, lat_info_te = laterality_maps(hte)
    log(f"laterality info: {lat_info_te}")
    slots_te = pick_slots(hte, plane_map)
    cov = pd.Series([len(v) for v in slots_te.values()]).describe()
    log(f"test slots per study: mean {cov['mean']:.2f} min {cov['min']:.0f} max {cov['max']:.0f}")
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_te, "test")

    fold_states = bundle.get("fold_states", [])
    if not fold_states:
        raise ValueError("saved model bundle has no fold_states")

    rank_sum = np.zeros((len(st_te), len(TARGETS)), np.float64)
    for k, fold in enumerate(fold_states, 1):
        model = build_model().to(dev)
        model.load_state_dict(fold["state_dict"])
        P = predict(model, Cte, Mte, np.arange(len(st_te)), dev)
        rank_sum += pd.DataFrame(P).rank(pct=True).values
        log(f"inferred fold {fold.get('fold', k - 1)} ({k}/{len(fold_states)})")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    sub = write_submission(rank_sum, len(fold_states), st_te, test_df)
    log(f"submission.csv {sub.shape}; models {len(fold_states)}")
    print(sub.head().to_string())


In [ ]:
try:
    main()
except Exception:
    traceback.print_exc()
    t = pd.read_csv(find_root() / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission_fallback.csv", index=False)
    print("wrote submission_fallback.csv; re-raising so this is not submit-ready")
    raise
log("done")
